# Notebook 03 — Studi Kasus 1: Klasifikasi

**Materi persiapan Ekshibisi Kecerdasan Artifisial · LKS Dikmen Nasional 2026**
Sasaran: siswa SMKN Bali Mandara, Program Keahlian Teknik Komputer dan Jaringan (TKJ)

Prasyarat: **Notebook 01 dan 02** sudah selesai.

---

## Apa itu klasifikasi

**Klasifikasi** adalah tugas menebak sebuah **label dari sejumlah pilihan yang
terbatas**. Contohnya:

| Masukan | Keluaran | Jumlah pilihan |
|---|---|---|
| foto sampah | organik / anorganik / B3 | 3 |
| teks pesan | hoaks / bukan hoaks | 2 |
| pengukuran tubuh pinguin | Adelie / Chinstrap / Gentoo | 3 |
| pola lalu lintas jaringan | normal / serangan | 2 |

Kalau keluarannya berupa **angka bebas** (harga, suhu, jumlah kWh), itu bukan
klasifikasi melainkan **regresi** — dibahas di Notebook 04.

## Kasus yang dikerjakan

Menebak spesies pinguin dari pengukuran tubuhnya. Dataset dan hasil EDA-nya
sudah kita kenal dari Notebook 02.

**Mengapa kasus ini yang dipakai untuk belajar?** Karena kita sudah tahu
jawabannya dari EDA: scatter plot di Notebook 02 menunjukkan ketiga spesies
hampir tidak bertumpuk. Jadi kalau akurasinya nanti rendah, kita tahu pasti
letaknya di kode — bukan di data. Untuk belajar, kasus yang hasilnya bisa diduga
lebih berguna daripada kasus yang sulit.

**Urutan kerja di bawah ini berlaku untuk kasus apa pun**, termasuk klasifikasi
jenis sampah atau pemeriksaan klaim hoaks pada studi kasus LKS 2026. Yang
berganti hanya datanya.

## Alur kerja lengkap

```
1. Muat data              -> pd.read_csv
2. Pahami data (EDA)      -> Notebook 02
3. Tentukan X dan y       -> fitur dan target
4. Bagi data              -> latih / uji     <- WAJIB, sebelum apa pun
5. Buat baseline          -> patokan terendah
6. Siapkan prapemrosesan  -> Pipeline
7. Latih beberapa model   -> .fit()
8. Bandingkan             -> cross-validation
9. Setel (tuning)         -> GridSearchCV
10. Uji SEKALI            -> data uji
11. Simpan & pakai        -> joblib
12. Catat risiko          -> Responsible AI
```

Estimasi waktu: 120–150 menit.

## Bagian 0 — Persiapan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

import sklearn, joblib

SEED = 42          # dipakai di semua tempat yang butuh angka acak
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

print("scikit-learn:", sklearn.__version__)
print("pandas      :", pd.__version__)

`SEED = 42` dipakai di setiap tempat yang melibatkan keacakan. Tanpa ini, setiap
kali notebook dijalankan hasilnya sedikit berbeda dan kamu tidak bisa
membandingkan dua percobaan secara adil. Angka 42 tidak istimewa — yang penting
sama di semua tempat.

---

## Bagian 1 — Muat data dan periksa ulang

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(URL)

print(f"{df.shape[0]} baris, {df.shape[1]} kolom")
print()
print("Nilai hilang per kolom:")
print(df.isna().sum())
print()
print("Sebaran target (species):")
print(df["species"].value_counts())
df.head()

### Keputusan penanganan data hilang

Dua baris kehilangan seluruh kolom pengukuran — tidak ada yang bisa dipelajari
dari baris itu, jadi dibuang. Kolom `sex` yang hilang pada 9 baris **tidak**
dibuang: baris-baris itu masih punya semua pengukuran tubuh yang berguna. Nilai
`sex`-nya akan diisi otomatis di dalam `Pipeline` nanti (Bagian 4).

Perhatikan bedanya: satu dibuang karena tidak punya informasi, satu dipertahankan
karena masih punya informasi. Ini keputusan, bukan aturan otomatis — dan harus
bisa kamu jelaskan.

In [ ]:
kolom_ukur = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

sebelum = len(df)
df = df.dropna(subset=kolom_ukur).reset_index(drop=True)

print(f"{sebelum} baris -> {len(df)} baris (dibuang {sebelum - len(df)}, "
      f"{(sebelum - len(df)) / sebelum * 100:.1f}%)")
print()
print("Sisa nilai hilang:")
print(df.isna().sum()[lambda s: s > 0])

---

## Bagian 2 — Menentukan `X` dan `y`

Dua istilah yang harus tepat:

- **`y` (target / label)** — yang ingin ditebak. Di sini: `species`.
- **`X` (fitur)** — bahan untuk menebak. Di sini: pengukuran tubuh + pulau + sex.

Huruf `X` besar karena berisi banyak kolom (tabel), `y` kecil karena satu kolom.
Kebiasaan penulisan ini dipakai di seluruh dokumentasi scikit-learn.

### Satu kolom yang perlu dipikirkan: `island`

Dari EDA Notebook 02 kita tahu Gentoo hanya ada di Biscoe dan Chinstrap hanya di
Dream. Memasukkan `island` akan menaikkan akurasi — tapi model itu belajar
"di mana", bukan "bagaimana bentuk tubuhnya".

Kalau model ini nanti dipakai di pulau lain, tebakannya rusak. Masalah semacam
ini disebut **pintasan** (*shortcut learning*): model menemukan jalan pintas yang
kebetulan berhasil di data latih tapi tidak mencerminkan sebab yang sebenarnya.

Kita tidak menebak mana yang lebih baik — kita **ukur keduanya** di Bagian 6.

In [ ]:
fitur_angka = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
fitur_kategori = ["island", "sex"]

X = df[fitur_angka + fitur_kategori]
y = df["species"]

print("Bentuk X:", X.shape)
print("Bentuk y:", y.shape)
print()
print("Kelas yang ada:", sorted(y.unique()))
print()
X.head()

---

## Bagian 3 — Membagi data latih dan data uji

Ini langkah yang paling sering dilakukan dengan salah, dan paling mahal
akibatnya.

### Mengapa harus dibagi

Model bisa "menghafal" data yang dilihatnya. Kalau diuji dengan data yang sama,
nilainya tinggi tapi tidak berarti apa-apa — seperti ujian dengan soal yang
kuncinya sudah dibagikan. Satu-satunya cara mengetahui kemampuan model yang
sebenarnya adalah mengujinya dengan data yang **belum pernah dilihatnya**.

### Tiga hal wajib pada `train_test_split`

| Argumen | Arti | Mengapa penting |
|---|---|---|
| `test_size=0.2` | 20% untuk uji | 20–30% umum dipakai |
| `random_state=SEED` | acak yang bisa diulang | tanpa ini hasilnya berubah terus |
| `stratify=y` | proporsi kelas dijaga sama | tanpa ini kelas kecil bisa habis di satu sisi |

`stratify=y` khususnya penting di sini: Chinstrap hanya 68 baris. Tanpa
stratifikasi, pembagian acak bisa menghasilkan data uji yang isinya Chinstrap
terlalu sedikit, dan hasil pengujiannya jadi tidak bisa dipercaya.

In [ ]:
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,          # jaga proporsi kelas di kedua sisi
)

print(f"Data latih : {len(X_latih)} baris")
print(f"Data uji   : {len(X_uji)} baris")
print()
perbandingan = pd.DataFrame({
    "latih_%": y_latih.value_counts(normalize=True).mul(100).round(1),
    "uji_%": y_uji.value_counts(normalize=True).mul(100).round(1),
    "latih_n": y_latih.value_counts(),
    "uji_n": y_uji.value_counts(),
})
print(perbandingan)

Proporsi di kedua kolom persen hampir identik — itulah kerja `stratify`.

> ### Aturan yang tidak boleh dilanggar
>
> **Sejak baris ini sampai Bagian 9, `X_uji` dan `y_uji` tidak boleh disentuh
> sama sekali.** Tidak untuk melihat sebarannya, tidak untuk menghitung
> rata-ratanya, tidak untuk memilih model.
>
> Begitu data uji ikut memengaruhi keputusanmu, ia berhenti menjadi data uji.
> Pelanggaran ini disebut **kebocoran data** (*data leakage*), dan akibatnya
> selalu sama: angka di laporan bagus, kenyataan di lapangan jelek.
>
> Tiga bentuk kebocoran yang paling sering terjadi tanpa disadari:
> 1. Menghitung rata-rata/skala dari seluruh data sebelum membaginya.
>    → dicegah dengan `Pipeline` (Bagian 4).
> 2. Mengisi nilai hilang memakai rata-rata seluruh data.
>    → dicegah dengan `SimpleImputer` di dalam `Pipeline`.
> 3. Memilih model berdasarkan skor data uji, lalu melaporkan skor itu juga.
>    → dicegah dengan cross-validation di data latih (Bagian 6).

---

## Bagian 4 — Prapemrosesan dan `Pipeline`

Model scikit-learn hanya menerima angka. Dua hal harus dibereskan dulu.

### a. Kolom kategori → angka: *one-hot encoding*

Kolom `island` berisi teks `Biscoe`, `Dream`, `Torgersen`. Diubah menjadi tiga
kolom berisi 0/1:

| island | island_Biscoe | island_Dream | island_Torgersen |
|---|---|---|---|
| Biscoe | 1 | 0 | 0 |
| Dream | 0 | 1 | 0 |

**Jangan** mengubahnya menjadi 1, 2, 3. Itu memberi tahu model bahwa Torgersen
"tiga kali" Biscoe dan bahwa Dream berada di antara keduanya — padahal ketiganya
hanya nama tempat, tidak punya urutan.

### b. Kolom angka → skala seragam: *standardisasi*

`body_mass_g` bernilai ribuan, `bill_depth_mm` belasan. Beberapa algoritma
(regresi logistik, KNN, jaringan saraf) menganggap angka besar lebih penting
semata-mata karena besarnya. `StandardScaler` menyetel setiap kolom agar
rata-ratanya 0 dan simpangannya 1, sehingga semua kolom setara.

Model berbasis pohon (`DecisionTree`, `RandomForest`) tidak terpengaruh skala.
Tapi memakai scaler tetap aman, jadi kita pakai satu susunan untuk semua.

### c. Mengapa `Pipeline`, bukan dikerjakan manual

`Pipeline` menggabungkan prapemrosesan dan model menjadi satu objek. Keuntungannya
bukan soal kerapian, tapi **kebenaran**:

- Saat `.fit(X_latih)`, rata-rata dan skala dihitung **hanya dari data latih**.
- Saat `.predict(X_uji)`, nilai yang sama itu diterapkan ke data uji.
- Cross-validation mengulang keduanya di setiap lipatan, otomatis.

Mengerjakannya manual bukan mustahil, tapi hampir selalu berakhir dengan
kebocoran yang tidak disadari. Pakai `Pipeline`.

In [ ]:
# Alur untuk kolom angka: isi yang hilang -> seragamkan skala
alur_angka = Pipeline([
    ("isi", SimpleImputer(strategy="median")),
    ("skala", StandardScaler()),
])

# Alur untuk kolom kategori: isi yang hilang -> ubah jadi 0/1
alur_kategori = Pipeline([
    ("isi", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# ColumnTransformer: kolom mana diperlakukan dengan alur mana
prapemrosesan = ColumnTransformer([
    ("angka", alur_angka, fitur_angka),
    ("kategori", alur_kategori, fitur_kategori),
])

prapemrosesan

`strategy="median"` dipilih untuk kolom angka karena median tidak terpengaruh
pencilan — lebih aman daripada `"mean"` pada data nyata.

`handle_unknown="ignore"` pada `OneHotEncoder` penting untuk pemakaian
sesungguhnya: kalau nanti muncul nama pulau baru yang tidak ada di data latih,
model tidak akan galat, hanya mengabaikannya.

Mari lihat hasilnya supaya tidak terasa seperti kotak hitam.

In [ ]:
# Hanya untuk melihat hasilnya — dilatih pada data latih saja
contoh = prapemrosesan.fit_transform(X_latih)
nama_kolom = prapemrosesan.get_feature_names_out()

print("Bentuk sebelum :", X_latih.shape, "(6 kolom, ada teks)")
print("Bentuk sesudah :", contoh.shape, "(semua angka)")
print()
print("Nama kolom hasil:")
for n in nama_kolom:
    print("  ", n)
print()
print("Tiga baris pertama setelah diproses (dibulatkan):")
print(pd.DataFrame(contoh[:3], columns=nama_kolom).round(2).to_string())

Perhatikan: 6 kolom menjadi 9 kolom. Kolom angka sekarang bernilai sekitar -2
sampai 2 (hasil standardisasi), kolom kategori menjadi 0 atau 1.

---

## Bagian 5 — Baseline: patokan terendah

Sebelum melatih model sungguhan, kita perlu tahu **berapa skor yang bisa dicapai
tanpa belajar apa pun**. `DummyClassifier(strategy="most_frequent")` selalu
menjawab kelas yang paling banyak.

Angka ini adalah garis batas. Model yang tidak mengalahkannya tidak berguna,
sekeren apa pun namanya. Melaporkan "akurasi 70%" tanpa menyebut baseline-nya
adalah pelaporan yang menyesatkan — dan pertanyaan yang gampang ditanyakan juri.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent", random_state=SEED)
baseline.fit(X_latih, y_latih)

skor_baseline = accuracy_score(y_latih, baseline.predict(X_latih))
print(f"Baseline (selalu menjawab kelas terbanyak): {skor_baseline:.3f}")
print(f"Kelas yang selalu dijawab                : {baseline.predict(X_latih[:1])[0]}")
print()
print(f"-> Model apa pun harus di atas {skor_baseline:.3f} untuk dianggap belajar sesuatu.")

---

## Bagian 6 — Melatih dan membandingkan model

Sekarang bagian yang ditunggu. Perhatikan bahwa isinya hanya dua baris kunci:
`.fit()` untuk melatih, `.predict()` untuk menebak.

### Empat algoritma yang kita bandingkan

| Algoritma | Cara kerjanya, sesingkat mungkin | Kelebihan | Kekurangan |
|---|---|---|---|
| **Logistic Regression** | mencari garis pemisah antar kelas | cepat, bisa dijelaskan | hanya garis lurus |
| **Decision Tree** | rangkaian pertanyaan ya/tidak | paling mudah dijelaskan | mudah menghafal |
| **Random Forest** | banyak pohon, hasilnya divoting | biasanya paling akurat | lebih sulit dijelaskan |
| **KNN** | melihat k data terdekat | sangat sederhana | lambat kalau data besar |

Tidak ada yang "terbaik" secara umum. Untuk data tabel berukuran kecil-menengah,
Random Forest hampir selalu menjadi pilihan awal yang wajar.

In [ ]:
# Satu model dulu, supaya alurnya jelas
model_lr = Pipeline([
    ("prep", prapemrosesan),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
])

model_lr.fit(X_latih, y_latih)               # <- PELATIHAN (training)

tebakan_latih = model_lr.predict(X_latih)     # <- PENEBAKAN (inference)

print(f"Akurasi pada data latih: {accuracy_score(y_latih, tebakan_latih):.3f}")
print(f"Baseline                : {skor_baseline:.3f}")
print()
print("Lima tebakan pertama :", list(tebakan_latih[:5]))
print("Jawaban sebenarnya   :", list(y_latih[:5]))

> Akurasi pada **data latih** selalu terlalu bagus — model sudah melihat data itu.
> Angka ini tidak boleh dilaporkan sebagai hasil. Kita butuh cara menilai yang
> lebih jujur, tapi tanpa menyentuh data uji.

### Cross-validation: menilai tanpa membuka data uji

Data latih dibagi 5 bagian. Model dilatih 5 kali; setiap kali, 4 bagian untuk
melatih dan 1 bagian untuk menilai, bergilir. Hasilnya 5 skor — rata-rata dan
sebarannya jauh lebih bisa dipercaya daripada satu angka.

`StratifiedKFold` memastikan proporsi kelas terjaga di setiap lipatan, sama
alasannya dengan `stratify` di Bagian 3.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

skor = cross_val_score(model_lr, X_latih, y_latih, cv=cv, scoring="accuracy")

print("Skor per lipatan:", np.round(skor, 3))
print(f"Rata-rata       : {skor.mean():.3f}")
print(f"Simpangan       : {skor.std():.3f}")
print()
print(f"Dilaporkan sebagai: {skor.mean():.3f} +/- {skor.std():.3f}")

Bentuk `0.987 +/- 0.014` adalah cara pelaporan yang benar. Simpangannya penting:
kalau besar, berarti skornya sangat bergantung pada pembagian data — tanda bahwa
datanya terlalu sedikit atau modelnya tidak stabil.

Sekarang keempat algoritma sekaligus.

In [ ]:
kandidat = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEED),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
}

hasil = []
for nama, algoritma in kandidat.items():
    pipa = Pipeline([("prep", prapemrosesan), ("model", algoritma)])
    s = cross_val_score(pipa, X_latih, y_latih, cv=cv, scoring="accuracy")
    hasil.append({"model": nama, "rata_rata": s.mean(), "simpangan": s.std(),
                  "terendah": s.min(), "tertinggi": s.max()})

tabel = pd.DataFrame(hasil).sort_values("rata_rata", ascending=False).reset_index(drop=True)
tabel.insert(0, "baseline", skor_baseline)
print(tabel.round(4).to_string(index=False))

Keempatnya jauh di atas baseline. Selisih antar model kecil dan sebagian masih di
dalam rentang simpangannya — artinya **tidak bisa disebut ada yang jelas lebih
baik** pada data sekecil ini. Menyatakan "model A lebih baik" padahal selisihnya
0,004 dengan simpangan 0,02 adalah kesimpulan yang tidak didukung angka.

Dalam keadaan seperti ini, pilih berdasarkan pertimbangan lain: kecepatan,
kemudahan menjelaskan, atau kebutuhan pemakaian. Untuk lomba, **model yang bisa
kamu jelaskan** lebih berharga daripada model yang skornya 0,3% lebih tinggi.

In [ ]:
# Grafik perbandingan, dengan garis baseline
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(tabel["model"], tabel["rata_rata"], xerr=tabel["simpangan"],
        color="#4C72B0", capsize=4)
ax.axvline(skor_baseline, color="#C44E52", linestyle="--",
           label=f"baseline {skor_baseline:.3f}")
ax.set_xlim(0, 1.05)
ax.set_xlabel("akurasi cross-validation (garis = simpangan)")
ax.set_title("Perbandingan model pada data latih")
ax.legend(loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Uji pertanyaan dari Bagian 2: apakah `island` membuat model menghafal lokasi?

Kita latih ulang tanpa kolom `island` dan bandingkan. Inilah cara menjawab
pertanyaan desain: diukur, bukan didebatkan.

In [ ]:
# Susunan tanpa island — hanya pengukuran tubuh + sex
prep_tanpa_pulau = ColumnTransformer([
    ("angka", alur_angka, fitur_angka),
    ("kategori", alur_kategori, ["sex"]),
])

pipa_dengan = Pipeline([("prep", prapemrosesan),
                        ("model", RandomForestClassifier(n_estimators=200, random_state=SEED))])
pipa_tanpa = Pipeline([("prep", prep_tanpa_pulau),
                       ("model", RandomForestClassifier(n_estimators=200, random_state=SEED))])

s_dengan = cross_val_score(pipa_dengan, X_latih, y_latih, cv=cv)
s_tanpa = cross_val_score(pipa_tanpa, X_latih, y_latih, cv=cv)

print(f"Dengan island : {s_dengan.mean():.4f} +/- {s_dengan.std():.4f}")
print(f"Tanpa island  : {s_tanpa.mean():.4f} +/- {s_tanpa.std():.4f}")
print(f"Selisih       : {s_dengan.mean() - s_tanpa.mean():+.4f}")

Selisihnya sangat kecil. Kesimpulannya: **kolom `island` hampir tidak dibutuhkan**
— pengukuran tubuh saja sudah cukup.

Maka pilihan yang lebih baik adalah membuangnya. Modelnya jadi lebih sederhana,
lebih mungkin tetap bekerja di lokasi lain, dan tidak bisa dituduh hanya menghafal
geografi. Kita lanjut dengan susunan **tanpa `island`**.

Ini contoh keputusan yang bagus untuk diceritakan saat tanya jawab: bukan
"saya pakai Random Forest karena paling akurat", tapi "saya membuang satu fitur
walaupun akurasinya turun sedikit, karena fitur itu membuat model bergantung pada
lokasi pengamatan."

---

## Bagian 7 — Overfitting: model yang menghafal

**Overfitting** terjadi ketika model menghafal data latih, termasuk gangguan
acaknya, sehingga gagal pada data baru. Tandanya: skor data latih tinggi, skor
cross-validation jauh lebih rendah.

Cara termudah melihatnya: pohon keputusan tanpa batas kedalaman.

In [ ]:
uji_kedalaman = []
for d in [1, 2, 3, 4, 6, 10, None]:
    pipa = Pipeline([
        ("prep", prep_tanpa_pulau),
        ("model", DecisionTreeClassifier(max_depth=d, random_state=SEED)),
    ])
    pipa.fit(X_latih, y_latih)
    skor_latih = accuracy_score(y_latih, pipa.predict(X_latih))
    skor_cv = cross_val_score(pipa, X_latih, y_latih, cv=cv).mean()
    uji_kedalaman.append({
        "max_depth": "tanpa batas" if d is None else d,
        "skor_latih": round(skor_latih, 4),
        "skor_cv": round(skor_cv, 4),
        "selisih": round(skor_latih - skor_cv, 4),
    })

print(pd.DataFrame(uji_kedalaman).to_string(index=False))

Baca kolom `selisih`:

- `max_depth=1` — kedua skor rendah. Model terlalu sederhana: **underfitting**.
- `max_depth=3` atau `4` — kedua skor tinggi dan berdekatan. Ini yang dicari.
- `tanpa batas` — skor latih 1,000 (hafal sempurna) tapi skor CV turun.
  **Overfitting**.

Selisih besar antara skor latih dan skor cross-validation adalah tanda peringatan
paling penting dalam machine learning. Biasakan selalu mencetak keduanya.

Pohon dengan kedalaman kecil juga punya keuntungan lain: bisa digambar dan
dibaca manusia.

In [ ]:
pohon = Pipeline([("prep", prep_tanpa_pulau),
                  ("model", DecisionTreeClassifier(max_depth=3, random_state=SEED))])
pohon.fit(X_latih, y_latih)

plt.figure(figsize=(15, 7))
plot_tree(pohon.named_steps["model"],
          feature_names=pohon.named_steps["prep"].get_feature_names_out(),
          class_names=pohon.named_steps["model"].classes_,
          filled=True, rounded=True, fontsize=8, impurity=False)
plt.title("Pohon keputusan, max_depth=3 — seluruh isi model ini bisa dibaca")
plt.tight_layout()
plt.show()

Inilah **seluruh isi** model itu: serangkaian pertanyaan ya/tidak atas nilai yang
sudah distandardisasi. Tidak ada yang tersembunyi.

Kemampuan menunjukkan gambar ini saat presentasi sangat berharga. Panduan LKS
2026 menilai "penguasaan teknis tim" pada sesi tanya jawab — dan tidak ada yang
lebih meyakinkan daripada bisa menjelaskan keputusan model satu per satu.

---

## Bagian 8 — Metrik: akurasi saja tidak cukup

Empat metrik yang perlu kamu pahami. Ambil satu kelas, misalnya Chinstrap:

| Metrik | Pertanyaannya | Rumus sederhananya |
|---|---|---|
| **Accuracy** | berapa persen seluruh tebakan benar? | benar / semua |
| **Precision** | dari yang ditebak Chinstrap, berapa yang benar? | benar Chinstrap / semua tebakan Chinstrap |
| **Recall** | dari semua Chinstrap sungguhan, berapa yang tertangkap? | benar Chinstrap / semua Chinstrap asli |
| **F1** | gabungan seimbang precision dan recall | — |

### Kapan mana yang penting — ini pertanyaan tentang akibat, bukan tentang rumus

- **Recall penting** bila melewatkan kasus positif berbahaya.
  Contoh: alat pemeriksa hoaks. Hoaks yang lolos lebih merugikan daripada berita
  benar yang kena tandai.
- **Precision penting** bila tuduhan palsu merugikan.
  Contoh: sistem yang otomatis menghapus unggahan. Menghapus konten yang
  sebenarnya benar merusak kepercayaan.

Keduanya tidak bisa dimaksimalkan bersama-sama; menaikkan yang satu biasanya
menurunkan yang lain. **Memilih mana yang diutamakan adalah keputusan manusia,
bukan keputusan model.** Pertimbangan inilah yang dinilai pada komponen
Responsible AI (bobot 15% pada panduan LKS 2026).

In [ ]:
# Laporan lengkap, dihitung dengan cross-validation di data latih
from sklearn.model_selection import cross_val_predict

model_terpilih = Pipeline([
    ("prep", prep_tanpa_pulau),
    ("model", RandomForestClassifier(n_estimators=200, random_state=SEED)),
])

tebakan_cv = cross_val_predict(model_terpilih, X_latih, y_latih, cv=cv)

print("Laporan klasifikasi (dari cross-validation, BUKAN data uji):")
print()
print(classification_report(y_latih, tebakan_cv, digits=3))

Cara membaca laporan itu:

- **`support`** — jumlah data asli tiap kelas. Selalu lihat ini dulu; metrik atas
  kelas dengan support 10 tidak bisa dipercaya sebesar kelas dengan support 100.
- **`macro avg`** — rata-rata antar kelas, semua kelas dianggap sama penting.
- **`weighted avg`** — rata-rata dengan bobot jumlah data. Kelas besar lebih
  berpengaruh.

Kalau kelasnya tidak seimbang, **laporkan `macro avg`**. `weighted avg` bisa
terlihat bagus walaupun kelas kecil selalu salah ditebak.

In [ ]:
# Confusion matrix: memperlihatkan JENIS kesalahannya, bukan hanya jumlahnya
cm = confusion_matrix(y_latih, tebakan_cv, labels=sorted(y.unique()))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

ConfusionMatrixDisplay(cm, display_labels=sorted(y.unique())).plot(
    ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Jumlah")

ConfusionMatrixDisplay(cm / cm.sum(axis=1, keepdims=True),
                       display_labels=sorted(y.unique())).plot(
    ax=ax[1], cmap="Blues", colorbar=False, values_format=".2f")
ax[1].set_title("Proporsi per baris (recall per kelas)")

plt.tight_layout()
plt.show()

print("Baris = jawaban sebenarnya, kolom = tebakan model.")
print("Diagonal = benar. Di luar diagonal = salah, dan ketahuan tertukar dengan apa.")

Ini sebabnya confusion matrix lebih berguna daripada satu angka akurasi: ia
memberi tahu **apa yang tertukar dengan apa**.

Pada kasus nyata, informasi itu langsung menentukan tindakan. Misalnya pada
aplikasi pemilahan sampah: kalau "botol plastik" sering tertukar dengan "botol
kaca", yang dibutuhkan adalah lebih banyak contoh foto kedua kelas itu — bukan
ganti algoritma. Confusion matrix mengarahkan usaha ke tempat yang tepat.

---

## Bagian 9 — Fitur mana yang paling berpengaruh

**Permutation importance** bekerja dengan cara yang mudah dijelaskan: acak isi
satu kolom, lalu lihat berapa banyak skornya turun. Turun banyak = kolom itu
penting.

Keunggulannya dibanding `feature_importances_` bawaan Random Forest: bisa dipakai
untuk model apa pun, dan diukur pada data yang tidak dipakai melatih.

In [ ]:
model_terpilih.fit(X_latih, y_latih)

# Diukur dengan cross-validation di data latih — data uji tetap belum disentuh.
# n_repeats=15 supaya hasilnya stabil (tiap kolom diacak 15 kali).
imp = permutation_importance(model_terpilih, X_latih, y_latih,
                            n_repeats=15, random_state=SEED, scoring="accuracy")

tabel_imp = (pd.DataFrame({
    "fitur": X_latih.columns,
    "penurunan_akurasi": imp.importances_mean,
    "simpangan": imp.importances_std,
}).sort_values("penurunan_akurasi", ascending=False).reset_index(drop=True))

print(tabel_imp.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(tabel_imp["fitur"], tabel_imp["penurunan_akurasi"],
        xerr=tabel_imp["simpangan"], color="#55A868", capsize=3)
ax.set_xlabel("penurunan akurasi bila kolom ini diacak")
ax.set_title("Permutation importance")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

Perhatikan bahwa `island` ikut terhitung di sini karena masih ada di `X_latih`,
tetapi susunan `prep_tanpa_pulau` mengabaikannya — jadi nilainya nol. Itu
pemeriksaan yang bagus: kalau kolom yang seharusnya diabaikan ternyata punya
nilai importance bukan nol, berarti ada yang salah pada susunan pipeline-mu.

Dua catatan penting tentang importance:

1. **Nilai kecil belum berarti tidak berguna.** Kalau dua kolom berisi informasi
   yang mirip (`flipper_length_mm` dan `body_mass_g` berkorelasi 0,87), mengacak
   salah satunya tidak banyak berpengaruh karena yang lain masih ada.
2. **Importance bukan sebab-akibat.** Kolom ini berguna untuk menebak, bukan
   berarti ia yang menyebabkan.

---

## Bagian 10 — Penyetelan (*tuning*) dengan `GridSearchCV`

*Hyperparameter* adalah pengaturan yang kita tentukan sebelum pelatihan
(`max_depth`, `n_estimators`, `n_neighbors`), bukan yang dipelajari model dari
data.

`GridSearchCV` mencoba semua kombinasi yang kita daftarkan, menilai setiap
kombinasi dengan cross-validation, lalu memilih yang terbaik. Semuanya di dalam
data latih — data uji masih aman.

Penulisan nama parameternya: `namastep__namaparameter`, dengan **dua** garis
bawah. `model__max_depth` berarti "parameter `max_depth` pada step bernama
`model`".

In [ ]:
ruang_pencarian = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [3, 5, 8, None],
    "model__min_samples_leaf": [1, 3],
}

pencarian = GridSearchCV(
    Pipeline([("prep", prep_tanpa_pulau),
              ("model", RandomForestClassifier(random_state=SEED))]),
    param_grid=ruang_pencarian,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,          # pakai semua inti prosesor
    return_train_score=True,
)

pencarian.fit(X_latih, y_latih)

print(f"Kombinasi yang dicoba : {len(pencarian.cv_results_['params'])}")
print(f"Skor CV terbaik       : {pencarian.best_score_:.4f}")
print("Pengaturan terbaik    :")
for k, v in pencarian.best_params_.items():
    print(f"   {k} = {v}")

In [ ]:
# Lima kombinasi teratas — supaya terlihat apakah selisihnya berarti
ringkas = (pd.DataFrame(pencarian.cv_results_)
           [["param_model__n_estimators", "param_model__max_depth",
             "param_model__min_samples_leaf", "mean_train_score",
             "mean_test_score", "std_test_score"]]
           .sort_values("mean_test_score", ascending=False)
           .head(5).round(4))
ringkas.columns = ["n_estimators", "max_depth", "min_samples_leaf",
                   "skor_latih", "skor_cv", "simpangan_cv"]
print(ringkas.to_string(index=False))

Lihat selisih antara `skor_latih` dan `skor_cv`, lalu bandingkan selisih skor CV
antar baris dengan kolom `simpangan_cv`.

Pada dataset sekecil ini, kombinasi teratas biasanya berbeda jauh lebih kecil
daripada simpangannya. Artinya: **tuning tidak memberi perbaikan yang berarti di
sini.** Itu bukan kegagalan — itu temuan. Kalau sudah begini, waktu lebih baik
dipakai untuk menambah data atau memperbaiki fitur, bukan untuk menyetel
parameter.

Kesalahan yang sering terjadi pada lomba: menghabiskan berjam-jam untuk tuning
yang menaikkan 0,2%, padahal datanya hanya 300 baris.

---

## Bagian 11 — Pengujian akhir: sekali saja

Sekarang, dan hanya sekarang, data uji dibuka.

Angka yang keluar dari sel ini adalah angka yang kamu laporkan. Kalau hasilnya
kurang memuaskan lalu kamu kembali menyetel model dan mengujinya lagi, data uji
itu sudah tidak murni lagi — dan angka keduanya tidak jujur.

In [ ]:
model_akhir = pencarian.best_estimator_

tebakan_uji = model_akhir.predict(X_uji)
akurasi_uji = accuracy_score(y_uji, tebakan_uji)

print("=" * 58)
print("HASIL AKHIR PADA DATA UJI")
print("=" * 58)
print(f"Jumlah data uji        : {len(y_uji)} baris")
print(f"Baseline               : {skor_baseline:.4f}")
print(f"Skor CV (data latih)   : {pencarian.best_score_:.4f}")
print(f"Akurasi data uji       : {akurasi_uji:.4f}")
print(f"Benar                  : {int(akurasi_uji * len(y_uji))} dari {len(y_uji)}")
print("=" * 58)
print()
print(classification_report(y_uji, tebakan_uji, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_estimator(model_akhir, X_uji, y_uji,
                                     cmap="Blues", colorbar=False, ax=ax)
ax.set_title(f"Confusion matrix — data uji ({len(y_uji)} baris)")
plt.tight_layout()
plt.show()

### Kalau akurasinya 1,000 — jangan langsung senang

Pada dataset ini akurasi data uji sangat mungkin mencapai 1,000, yaitu 69 benar
dari 69. Itu hasil yang jujur, tetapi **bukan** berarti modelnya tidak akan
pernah salah. Yang benar-benar diketahui hanyalah: dalam 69 percobaan, tidak ada
yang salah.

Ada aturan praktis untuk menerjemahkannya: kalau dari `n` percobaan tidak ada
yang salah, tingkat kesalahan sebenarnya masih bisa sekitar `3/n`. Untuk 69 data
uji, itu berarti kesalahan sesungguhnya masih mungkin sampai sekitar 4%. Datanya
terlalu sedikit untuk menyimpulkan lebih dari itu.

Cara menyebutkannya dengan benar: **"69 dari 69 benar pada data uji"** — bukan
"akurasi 100%". Bentuk pertama menyampaikan sekaligus bahwa datanya hanya 69.

Skor sempurna juga layak dicurigai. Tiga hal yang wajib dicek lebih dulu:
apakah ada kolom yang mengandung jawaban, apakah data uji benar-benar terpisah,
dan apakah masalahnya memang mudah. Di kasus ini penyebabnya yang ketiga — dan
kita sudah tahu itu dari scatter plot di Notebook 02, sebelum melatih apa pun.

### Cara melaporkan yang benar

> Random Forest (300 pohon, `max_depth` hasil pencarian) dengan fitur pengukuran
> tubuh dan jenis kelamin. Akurasi cross-validation 5-lipat pada data latih
> (273 baris): sekitar 0,98. Akurasi pada data uji yang tidak pernah dipakai
> (69 baris): lihat angka di atas. Baseline kelas mayoritas: 0,44. Kolom `island`
> sengaja tidak dipakai agar model tidak bergantung pada lokasi pengamatan.

Tiga hal yang membuat laporan ini bisa dipercaya: **jumlah data disebutkan**,
**baseline disebutkan**, dan **keputusan desain dijelaskan beserta alasannya**.

Bandingkan dengan "model kami akurasinya 98%". Kalimat itu tidak bisa diperiksa
siapa pun.

---

## Bagian 12 — Menyimpan model dan memakainya

Model yang sudah dilatih bisa disimpan ke berkas, lalu dipakai ulang tanpa
melatih lagi. Karena yang kita simpan adalah seluruh `Pipeline`, prapemrosesannya
ikut tersimpan — data baru boleh masuk dalam bentuk aslinya.

In [ ]:
joblib.dump(model_akhir, "model_pinguin.joblib")
print("Model disimpan ke model_pinguin.joblib")

# Memuat ulang, seolah-olah di program lain
model_dimuat = joblib.load("model_pinguin.joblib")

# Data baru: bentuknya HARUS sama dengan X saat pelatihan (nama & jumlah kolom)
data_baru = pd.DataFrame([
    {"bill_length_mm": 39.1, "bill_depth_mm": 18.7, "flipper_length_mm": 181,
     "body_mass_g": 3750, "island": "Torgersen", "sex": "MALE"},
    {"bill_length_mm": 49.5, "bill_depth_mm": 15.0, "flipper_length_mm": 222,
     "body_mass_g": 5800, "island": "Biscoe", "sex": "MALE"},
    {"bill_length_mm": 46.5, "bill_depth_mm": 17.9, "flipper_length_mm": 192,
     "body_mass_g": 3500, "island": "Dream", "sex": "FEMALE"},
    # Baris ini sengaja dibuat berada di perbatasan Adelie dan Chinstrap
    {"bill_length_mm": 42.5, "bill_depth_mm": 18.5, "flipper_length_mm": 190,
     "body_mass_g": 3800, "island": "Dream", "sex": "FEMALE"},
])

tebakan = model_dimuat.predict(data_baru)
peluang = model_dimuat.predict_proba(data_baru)

for i, t in enumerate(tebakan):
    p = dict(zip(model_dimuat.classes_, peluang[i].round(3).tolist()))
    print(f"\nBaris {i + 1}: tebakan = {t}")
    print(f"   keyakinan: {p}")

### `predict_proba` — angka yang sering dilupakan, padahal paling berguna

`predict` memberi satu jawaban. `predict_proba` memberi **seberapa yakin** model
atas setiap kemungkinan.

Ini pintu masuk untuk menempatkan manusia di dalam sistem: bila keyakinan
tertinggi di bawah ambang tertentu, jangan diputuskan otomatis — minta manusia
memeriksanya.

In [ ]:
AMBANG = 0.70      # angka ini keputusan desain, bukan hasil perhitungan

for i, t in enumerate(tebakan):
    keyakinan = peluang[i].max()
    if keyakinan >= AMBANG:
        print(f"Baris {i + 1}: {t} (keyakinan {keyakinan:.2f}) -> diputuskan otomatis")
    else:
        print(f"Baris {i + 1}: {t} (keyakinan {keyakinan:.2f}) -> DITERUSKAN KE MANUSIA")

Pola sepuluh baris di atas adalah wujud paling konkret dari *human-in-the-loop*.
Kalau ada satu hal dari notebook ini yang sebaiknya masuk ke aplikasi lombamu,
ini pilihannya: murah, mudah dijelaskan, dan langsung menjawab pertanyaan juri
"di mana peran penilaian manusia dalam sistem ini?".

Cara menentukan ambangnya: lihat sebaran `predict_proba` pada data uji, lalu
pilih nilai yang membuat jumlah kasus yang diteruskan ke manusia masih bisa
ditangani. Ini keputusan operasional, bukan keputusan matematis.

---

## Bagian 13 — Responsible AI

Panduan LKS 2026 memberi bobot **15%** untuk komponen ini: minimal satu risiko
yang teridentifikasi, strategi mitigasinya, dan penjelasan di mana penilaian
manusia tetap berperan. Komponen ini sering diremehkan peserta, padahal bobotnya
setara dengan fungsionalitas aplikasi.

### Risiko pada model yang baru saja kita buat

| # | Risiko | Mengapa nyata | Mitigasi |
|---|---|---|---|
| 1 | **Data hanya dari 3 pulau, 1 wilayah, 1 periode** | model belum tentu berlaku di tempat lain | sebutkan batasan wilayah pada dokumentasi; jangan klaim berlaku umum |
| 2 | **Pintasan geografis** | `island` hampir menentukan spesies | kolom `island` dibuang; sudah diukur bahwa akurasinya tidak turun berarti |
| 3 | **Data uji hanya 69 baris** | satu tebakan salah menggeser akurasi ~1,4% | laporkan jumlah data uji dan simpangan CV, bukan satu angka |
| 4 | **Keyakinan rendah diperlakukan sama dengan tinggi** | tebakan meragukan bisa dianggap pasti | ambang `predict_proba` + penerusan ke manusia (Bagian 12) |
| 5 | **Kolom `sex` ada 9 nilai hilang** | diisi modus, bisa memperkuat kelompok mayoritas | dicatat di dokumentasi; alternatifnya kolom ini tidak dipakai |

### Cara memindahkan tabel ini ke kasus lombamu

Ganti isinya, pertahankan bentuknya. Untuk aplikasi klasifikasi sampah misalnya:

- **Risiko** — foto latih semuanya berlatar terang di dalam ruangan, sedangkan
  pemakaian nyata di tempat sampah luar ruangan yang gelap.
- **Mitigasi** — kumpulkan foto pada kondisi cahaya yang beragam; laporkan
  akurasi terpisah untuk kondisi terang dan gelap.
- **Peran manusia** — bila keyakinan di bawah 70%, aplikasi menampilkan
  "tidak yakin, mohon periksa" dan tidak memberi keputusan.

Jangan menulis risiko yang mengambang seperti "AI bisa salah". Sebutkan **risiko
yang spesifik pada datamu**, cara kamu mengukurnya, dan apa yang kamu lakukan.
Itu yang membedakan jawaban yang meyakinkan dari jawaban hafalan.

### Batasan yang harus disebutkan tentang dataset ini

Data pinguin ini dikumpulkan di Palmer Station, Antarktika, pada 2007–2009.
Model yang dilatih darinya tidak berlaku untuk spesies pinguin lain, tidak
berlaku di luar wilayah itu, dan tidak boleh dipakai sebagai alat pengambil
keputusan konservasi tanpa pemeriksaan ahli.

---

## Bagian 14 — Cek pemahaman

In [ ]:
# ---- Kerjakan ----

# a) Berapa baris data latih dan data uji?
n_latih, n_uji = None, None

# b) Berapa nilai baseline (akurasi kelas mayoritas)? Bulatkan 2 desimal.
baseline_bulat = None

# c) Latih Pipeline berisi KNeighborsClassifier(n_neighbors=7) memakai
#    prep_tanpa_pulau, lalu hitung rata-rata cross_val_score pada data latih.
#    Simpan hasilnya (float) ke skor_knn7.
skor_knn7 = None

# d) Berapa banyak tebakan yang SALAH pada data uji? (integer)
salah_uji = None


# ---- Pemeriksaan otomatis ----
assert (n_latih, n_uji) == (273, 69), f"(a) seharusnya (273, 69), dapatnya {(n_latih, n_uji)}"
assert baseline_bulat == 0.44, f"(b) seharusnya 0.44, dapatnya {baseline_bulat}"
assert skor_knn7 is not None and skor_knn7 > 0.90, \
    f"(c) skor KNN k=7 seharusnya di atas 0.90, dapatnya {skor_knn7}"
assert salah_uji is not None and salah_uji == (y_uji != tebakan_uji).sum(), \
    f"(d) jumlah tebakan salah belum tepat, dapatnya {salah_uji}"

print("Semua pemeriksaan lolos.")

---

## Bagian 15 — Latihan mandiri: ganti datanya

Alur di notebook ini tidak terikat pada pinguin. Kerjakan ulang seluruhnya
memakai data konsumsi listrik sekolah yang ada pada paket materi, di
`03-praktikum/pengayaan-python/konsumsi-listrik-sekolah-sintetis.csv`.

Unggah berkasnya ke Colab (Notebook 01 Bagian 8), lalu:

1. Muat dengan `pd.read_csv("konsumsi-listrik-sekolah-sintetis.csv")`.
2. Lakukan EDA sesuai delapan pertanyaan di Notebook 02 Bagian 4.
3. Buat kolom target baru:
   ```python
   df["boros"] = (df["kwh"] / df["jam_pakai"] > 1.2).map({True: "boros", False: "wajar"})
   ```
   Sesuaikan angka 1.2 setelah melihat sebaran datanya.
4. Tentukan `X` (misalnya `jenis_ruang`, `hari`, `jam_pakai`) dan `y` = `boros`.
5. **Hati-hati:** jangan memasukkan `kwh` ke dalam `X`. Target dihitung dari
   `kwh`, jadi memasukkannya berarti memberi jawaban kepada model — bentuk
   kebocoran yang paling sering terjadi pada kolom buatan sendiri.
6. Bagi data, buat baseline, bandingkan model, evaluasi dengan confusion matrix.
7. Tulis tabel risiko Responsible AI untuk kasus ini. Petunjuk: datanya sintetis,
   bukan pengukuran nyata — itu risiko pertama yang wajib disebutkan.

**Latihan tambahan.** Tulis tiga kalimat yang akan kamu ucapkan kepada dewan juri
untuk menjelaskan hasil modelmu. Sebutkan angkanya, jumlah datanya, dan
baseline-nya.

---

## Ringkasan

### Kerangka kode yang bisa dipakai ulang

```python
# 1. Muat dan bersihkan
df = pd.read_csv(URL).dropna(subset=kolom_wajib)

# 2. Pisahkan fitur dan target
X, y = df[fitur_angka + fitur_kategori], df[target]

# 3. Bagi — sebelum apa pun yang lain
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Baseline
DummyClassifier(strategy="most_frequent").fit(X_latih, y_latih)

# 5. Pipeline: prapemrosesan + model dalam satu objek
pipa = Pipeline([("prep", ColumnTransformer([...])), ("model", ...)])

# 6. Nilai dengan cross-validation di data latih
cross_val_score(pipa, X_latih, y_latih, cv=StratifiedKFold(5, shuffle=True,
                                                           random_state=42))

# 7. Setel
GridSearchCV(pipa, param_grid, cv=cv).fit(X_latih, y_latih)

# 8. Uji SEKALI
accuracy_score(y_uji, model_akhir.predict(X_uji))

# 9. Simpan
joblib.dump(model_akhir, "model.joblib")
```

### Sepuluh kesalahan yang paling sering terjadi

| Kesalahan | Akibatnya |
|---|---|
| Tidak membagi data latih/uji | skor tinggi yang tidak berarti apa pun |
| Prapemrosesan sebelum pembagian | kebocoran data |
| Lupa `stratify` pada data tak seimbang | kelas kecil bisa habis di satu sisi |
| Tidak punya baseline | tidak tahu model benar-benar belajar atau tidak |
| Melaporkan skor data latih | angka yang terlalu bagus |
| Menguji data uji berkali-kali | data uji tidak lagi murni |
| Hanya melaporkan akurasi | kesalahan pada kelas kecil tersembunyi |
| Tidak menyebut jumlah data | pembaca tidak bisa menilai keandalannya |
| Fitur yang mengandung jawaban | skor hampir sempurna, di lapangan gagal |
| Lupa `random_state` | hasil tidak bisa diulang |

### Berikutnya

**Notebook 04 — Studi Kasus Regresi.** Alur kerjanya sama, tetapi targetnya
berupa angka bebas. Yang berganti: metriknya, cara membaca hasilnya, dan
diagnosis kesalahannya.